# Generate SHORT, length-matched AI reviews (OPEN models)

Fixes a length mismatch in the multi-domain CG/OR dataset: the earlier AI generation
made only LONG reviews (~190 words), but real **app/game** reviews are very short
(median 16/21 words), so length-matching dropped most of those domains.

This generates AI reviews in **length buckets weighted to match each domain's real
length distribution** (mostly tiny/short for apps & games; a spread for
books/restaurants/products). Movies is skipped — its real reviews are long, already matched.

Open models only (Qwen/Llama/Zephyr). Short reviews generate fast.

### Instructions
1. **Runtime -> GPU** (H100/A100/G4; on T4 set `LOAD_4BIT=True`).
2. Run all cells (weights freed between models, disk won't fill).
3. Download `ai_short_reviews_open.csv` -> put in project `data/` and tell the assistant.

In [ ]:
!pip install -q -U transformers accelerate bitsandbytes

In [ ]:
import os, csv, random, gc, shutil
import torch
from transformers import pipeline, BitsAndBytesConfig

SEED = 42; random.seed(SEED)
MODELS = [("Qwen/Qwen2.5-7B-Instruct","qwen"),
          ("NousResearch/Meta-Llama-3-8B-Instruct","llama3"),
          ("HuggingFaceH4/zephyr-7b-beta","zephyr")]
PER_DOMAIN_PER_MODEL = 800     # 5 domains x 3 models x 800 ~ 12,000 (mostly short -> fast)
BATCH     = 32                 # H100/G4; lower to 16 on A100, 8 on T4
LOAD_4BIT = False              # True on T4
OUT       = "ai_short_reviews_open.csv"

# length bucket -> (instruction, max_new_tokens). Tiny/short fill the gap the long gen left.
LENGTH_BUCKETS = {
    "tiny":   ("in one very short sentence (a quick reaction, about 5-15 words)", 40),
    "short":  ("in one or two brief sentences (about 15-40 words)",              80),
    "medium": ("in a short paragraph (about 40-90 words)",                       170),
    "long":   ("in a detailed paragraph (about 120-200 words)",                  300),
}
# per-domain bucket weights ~ matched to the real human length distribution
DOMAIN_BUCKETS = {
    "apps":        {"tiny":0.55,"short":0.30,"medium":0.13,"long":0.02},
    "games":       {"tiny":0.45,"short":0.28,"medium":0.20,"long":0.07},
    "books":       {"tiny":0.10,"short":0.25,"medium":0.30,"long":0.35},
    "restaurants": {"tiny":0.10,"short":0.25,"medium":0.40,"long":0.25},
    "products":    {"tiny":0.08,"short":0.32,"medium":0.45,"long":0.15},
}
DOMAINS = {
    "apps":  ("you've been using", ["a budgeting app","a fitness-tracking app","a photo-editing app",
              "a language-learning app","a meditation app","a food-delivery app","a note-taking app",
              "a ride-share app","a dating app","a streaming app","a weather app","a mobile game app"]),
    "games": ("you played", ["an open-world RPG","an indie platformer","a first-person shooter",
              "a cozy farming sim","a roguelike","a racing game","a puzzle game","a survival-horror game",
              "a strategy game","a fighting game","a mobile gacha game","a battle-royale shooter"]),
    "books": ("you read", ["a mystery novel","a fantasy epic","a self-help book","a historical romance",
              "a sci-fi novel","a memoir","a literary thriller","a cookbook","a young-adult novel",
              "a non-fiction history book","a thriller","a biography"]),
    "restaurants": ("you visited", ["an Italian restaurant","a sushi spot","a taco truck","a steakhouse",
              "a vegan cafe","a burger joint","a Thai restaurant","a brunch place","a pizzeria","a diner",
              "a ramen shop","a coffee shop","a Mexican cantina","an Indian restaurant"]),
    "products": ("you bought", ["wireless earbuds","a kitchen blender","a phone case","a coffee maker",
              "a backpack","a skincare cream","a kids' toy","a power drill","a vacuum cleaner","a desk lamp",
              "a water bottle","a pair of running shoes","a board game","a set of bed sheets"]),
}
PERSONAS = ["a busy parent","a college student","a retiree","a casual user","a power user","a first-timer",
            "a longtime fan","a skeptic","a bargain hunter","a tourist","a picky buyer"]
STYLES   = ["polished and well-written","casual and conversational",
            "quick and informal, all lowercase with a couple of typos like a phone review",
            "enthusiastic","measured and a little critical","blunt and to the point"]
STARS    = [5,5,5,5,4,4,4,3,2,1,1]

In [ ]:
def pick_bucket(domain):
    buckets = list(DOMAIN_BUCKETS[domain].keys())
    weights = list(DOMAIN_BUCKETS[domain].values())
    return random.choices(buckets, weights=weights, k=1)[0]

def build_prompt(domain, bucket):
    verb, subjects = DOMAINS[domain]
    subject = random.choice(subjects); persona = random.choice(PERSONAS)
    style = random.choice(STYLES); stars = random.choice(STARS)
    instr = LENGTH_BUCKETS[bucket][0]
    user = (f"Write a realistic online review of {subject} {verb}, as if you are {persona}, "
            f"giving {stars} out of 5 stars. Write it {instr}, {style}. Sound like a genuine human - "
            f"no header, no preamble, no star rating. Just output the review text.")
    return user, persona, style, stars, subject

In [ ]:
# resume-safe per-generator count
have = {fam: 0 for _, fam in MODELS}
if os.path.exists(OUT):
    import pandas as pd
    try:
        for s, c in pd.read_csv(OUT)["source"].value_counts().items():
            if s in have: have[s] = int(c)
    except Exception: pass
print("existing per generator:", have)

new = not os.path.exists(OUT)
f = open(OUT, "a", newline="", encoding="utf-8"); w = csv.writer(f)
if new: w.writerow(["text","domain","source","model_id","rating","persona","style","subject","length_bucket"])
TARGET_PER_MODEL = PER_DOMAIN_PER_MODEL * len(DOMAINS)

for model_id, fam in MODELS:
    if have[fam] >= TARGET_PER_MODEL:
        print(f"== {fam}: already has {have[fam]}, skipping =="); continue
    print(f"\n=== loading {model_id} ===")
    kw = dict(torch_dtype=torch.float16, device_map="auto", trust_remote_code=True)
    if LOAD_4BIT:
        kw = dict(model_kwargs={"quantization_config": BitsAndBytesConfig(
                      load_in_4bit=True, bnb_4bit_compute_dtype=torch.float16)},
                  device_map="auto", trust_remote_code=True)
    try:
        pipe = pipeline("text-generation", model=model_id, **kw)
    except Exception as e:
        print("  load failed, skipping:", e); continue
    tok = pipe.tokenizer
    if tok.pad_token_id is None: tok.pad_token_id = tok.eos_token_id
    tok.padding_side = "left"

    made = 0
    for domain in DOMAINS:
        # group by bucket so each batch shares one max_new_tokens
        for bucket, frac in DOMAIN_BUCKETS[domain].items():
            count = round(PER_DOMAIN_PER_MODEL * frac)
            if count <= 0: continue
            max_tok = LENGTH_BUCKETS[bucket][1]
            prompts, meta = [], []
            for _ in range(count):
                user, persona, style, stars, subject = build_prompt(domain, bucket)
                prompts.append([{"role":"user","content":user}]); meta.append((stars,persona,style,subject))
            for i in range(0, len(prompts), BATCH):
                batch = prompts[i:i+BATCH]
                try:
                    outs = pipe(batch, max_new_tokens=max_tok, do_sample=True, temperature=0.9,
                                top_p=0.95, batch_size=len(batch), pad_token_id=tok.pad_token_id)
                except Exception as e:
                    print("  gen error:", e); continue
                for j, o in enumerate(outs):
                    rec = o[0] if isinstance(o, list) else o
                    gen = rec["generated_text"]
                    text = gen[-1]["content"] if isinstance(gen, list) else str(gen)
                    text = text.strip().replace("\n", " ")
                    stars, persona, style, subject = meta[i+j]
                    if len(text.split()) >= 3:
                        w.writerow([text, domain, fam, model_id, stars, persona, style, subject, bucket]); made += 1
            f.flush(); print(f"  {fam} {domain}/{bucket}: total {made}")
    print(f"== {fam}: +{made} ==")
    del pipe; gc.collect(); torch.cuda.empty_cache()
    shutil.rmtree(os.path.expanduser("~/.cache/huggingface/hub/models--" +
                  model_id.replace("/", "--")), ignore_errors=True)
    print(f"   {fam} weights cache freed")

f.close()
print("done ->", OUT)

In [ ]:
import pandas as pd
df = pd.read_csv(OUT)
wl = df["text"].str.split().str.len()
print("total:", len(df), "| per generator:", df["source"].value_counts().to_dict())
print("median words:", int(wl.median()), "| %<=40w:", round((wl<=40).mean()*100), "%")
print("by domain median words:")
for d,g in df.groupby("domain"):
    print(f"  {d:12} n={len(g):>5} median={int(g['text'].str.split().str.len().median()):>4}")
from google.colab import files
files.download(OUT)